# trial_00 — experiment_1 protocol anchor (⚠️ READ FIRST)

This notebook is the **contract** for every trial in experiment_1. It fixes the benchmark, the objective, the search space, the runtime budget, and the structural choices. A trial's job is to pick hyperparameters and beat the baseline — not to change anything on this page. Changes to the protocol need **Anuar's OK** (see `instructions.md`).

Experiment_1 = algorithm **TPE** · model **KDE+REINFORCE inversion, Lorentzian-width (TPA)** · benchmark **synthetic**.

## 1. The question

Minimize the recovery error of the joint (μ, γ) optimizer by tuning its **8 hyperparameters**:

`n_runs, n_iter, lr_mu, lr_gamma, sigma_ref, clip, gamma_anneal, h_s_min`

The optimizer itself (score forms, likelihood, line shape) is **fixed for this campaign** — structural changes are a different conversation (propose → Anuar).

## 2. Benchmark (fixed)

- **14 synthetic experiments**: 2 powers (1nW, 3nW) × 7 temperatures (Trans05…Trans100), true params from Gregor's fits (16-series).
- Targets are generated **at true values** with `SYNTH_SEED=12345` → **identical targets across all trials** (fair comparisons).
- Runs are **deterministic** (`SEED=42` per-step noise) → the only noise in the objective is the sampling spread across experiments.

Full list lives in `ag_hypopt.py` → `EXPERIMENTS`. A reduced subset can be enabled via `BENCHMARK_SUBSET` (open decision, §9).

## 3. Objective (fixed)

- Per experiment: **relative squared error** for μ and for γ (each normalized by its true value).
- **objective** = mean over all 2×14 = 28 per-experiment errors (lower is better).
- **uncertainty** = SE of those 28 errors (sampling uncertainty across experiments — Anuar, 2026-08-29 11:15).
- **No Fisher** in the objective (optimization only).
- Implemented in `ag_hypopt.compute_objective()` — the metric never drifts between trial copies.

## 4. Search space (tunable — this is what the agent tunes)

| param | type | range | role |
|---|---|---|---|
| n_runs | int | 100–500 | runs per optimization step (runtime driver) |
| n_iter | int | 100–400 | optimization steps (runtime driver) |
| lr_mu | float | 5–40 | REINFORCE μ learning rate (linear decay, no floor) |
| lr_gamma | float | 0.1–1.5 | γ learning rate (annealed ×(1−γ_anneal·t/N)) |
| sigma_ref | float | 5–25 | μ-score normalization (scale-invariant step) |
| clip | float | 5–20 | gradient clipping |
| gamma_anneal | float | 0–0.75 | γ anneal strength (0 = none, 17e style) |
| h_s_min | float | 0–0.2 | σ-kernel bandwidth floor (MHz) |

Defined in `space.json` (parameters + dependencies, consumed by the `AGHyperopt` class). **Provisional** — to be revised after the model analysis (which variables survive may change).

## 5. Fixed structural choices (NOT tunable this campaign)

- **z-form γ-score**: dlogG = (d_f·∂F/∂γ/H_F + d_s·∂σ/∂γ/H_S)/H_REF, H_REF=1.0 — the 17g fix (γ RMSE 2.27→0.59 on synthetic)
- **σ_ref μ-score** (self-normalized, scale-invariant)
- **LAMBDA_MEAN = 0** (mean-matching anchor disabled)
- μ LR **linear decay with no floor**; γ update has the anneal schedule (strength = tunable)
- μ ∈ [1,200], γ ∈ [0.1,100] clamping
- 4 workers, fork pool, single-threaded per worker

## 6. Runtime budget

- Cap: **n_runs × n_iter ≤ 40000** (~3.5 h on the full benchmark at measured 17f speed).
- The baseline config sits **exactly at the cap** — the budget was chosen so the baseline is evaluable.
- Faster trials = reduced benchmark or tighter space (open decision §9).

## 7. Current state — the lineage that produced this protocol

- **16-series**: original design (KDE likelihood, REINFORCE μ, KDE γ).
- **17-series** (synthetic): σ_ref μ-score (17f, μ RMSE 1.40); **z-form γ-score (17g = BIG WIN: γ RMSE 2.27 → 0.59, μ 1.13)**; γ anneal backfired for γ in 17f (2.27) — cured by z-form.
- **18-series** (real data): μ lands ~½ true within huge σ_μ → weak identifiability (flat μ likelihood); 1nW γ systematically low even with z-form → **model mismatch** (Voigt/Lorentzian spread, failure mode #6), not optimizer; 3nW low-T unstable. → model work is a *separate* track (later); the campaign tunes the optimizer on synthetic.
- **Baseline to beat (baseline_17g, recorded in trials.json)**: objective = **0.001578 ± 0.000859** (μ rel-RMSE 3.6 %, γ rel-RMSE 4.3 %).

Remaining synthetic weaknesses worth attacking: 3nW T05 γ outlier (+2.14), μ bias at 3nW high-T.

In [ ]:
# Protocol summary — machine-readable, always current
import json, os
NOTEBOOK_DIR = os.getcwd()

data = json.load(open(os.path.join(NOTEBOOK_DIR, 'trials.json')))
protocol = data['protocol']
print('experiment:', protocol['experiment'])
print('benchmark:', protocol['benchmark'])
print('objective:', protocol['objective'])
print('runtime cap:', protocol['runtime_cap'])
print('space:')
space = json.load(open(os.path.join(NOTEBOOK_DIR, 'space.json')))['parameters']
for k, s in space.items():
    print(f"  {k:<14} {s['type']:<6} {s.get('low')} .. {s.get('high')}")
print('fixed structural:', '; '.join(protocol['fixed_structural']))
print('authority:', protocol['authority'])
best = min((t for t in data['trials'] if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print('registry:', len(data['trials']), 'trial(s) | best:',
      best['trial_id'] if best else None,
      f"(obj={best['objective']:.6f} ± {best['uncertainty']:.6f})" if best else "")


## 8. How to run a trial

1. Copy `template.ipynb` → `trial_XXX.ipynb` (next free number).
2. Follow it: read this anchor + all previous trials → propose (TPE+EI) → analyze with physics → choose → run (~3.5 h, in place) → analyze → record in `trials.json`.
3. `trials.json` is the single source of truth; `instructions.md` is the recipe card.

## 9. Open protocol decisions (Anuar)

- **Budget**: 3.5 h/trial vs reduced benchmark (~1 h) vs tighter space.
- **Space revision** after the model analysis (which variables/structures are actually tunable).
- Baseline re-evaluation: rerun baseline_17g through the harness for a same-machinery anchor point (recommended before trial_001).